
# Scenario 31 — preprocessing the standalone release

Every generalisation claim in this project rests on **50** labelled scenario-31
samples. This notebook converts the standalone release into **6,804** AMBER
samples, which is the single highest-value data task available to us.

Scenario 31 is held-out evaluation data. It is **never trained on** — doing so
would destroy the unseen-scenario claim it exists to test.

**Runtime: hours, CPU-bound.** Radar 2D-FFTs and LiDAR BEV histograms over ~7k
frames dominate. Internet must be on (the code is cloned).


## 1. Verify the release before doing hours of work

In [ ]:
import os, sys, subprocess
from pathlib import Path

RAW = Path("/kaggle/input/deepsense6g/scenario31")
assert RAW.is_dir(), f"{RAW} not found -- attach the deepsense6g dataset"

# The expensive failure is discovering after four hours of radar FFTs that a
# modality was never shipped. Check first, cheaply.
expect = {
    "camera": RAW / "unit1/camera_data",
    "lidar":  RAW / "unit1/lidar_data",
    "radar":  RAW / "unit1/radar_data",
    "power":  RAW / "unit1/mmWave_data",
    "gps_bs": RAW / "unit1/GPS_data",
    "gps_ue": RAW / "unit2/GPS_data",
}
print(f"{'modality':<10}{'present':>9}{'files':>9}")
missing = []
for k, p in expect.items():
    n = len(list(p.iterdir())) if p.is_dir() else 0
    print(f"{k:<10}{str(p.is_dir()):>9}{n:>9}")
    if not p.is_dir():
        missing.append(k)
csv = RAW / "scenario31_dev.csv"
print(f"\nlabels csv: {csv.exists()}  ({csv.stat().st_size/1e6:.1f} MB)" if csv.exists()
      else "\nlabels csv MISSING")
if missing:
    print(f"\nMISSING: {missing}")
    print("The pipeline tolerates a missing modality -- those samples get an"
          " availability-mask zero -- but configurations using it cannot be"
          " scored on scenario 31. Continuing.")

## 2. Code

In [ ]:
PROJECT = Path("/kaggle/working/project")
if not (PROJECT / "amber" / "model.py").exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git", str(PROJECT)],
                       capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    assert r.returncode == 0, (r.stderr or "")[:400]
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("code at", PROJECT)


## 3. Stage into the layout the pipeline expects

`config.SOURCES["scenario31_full"]` points at `data/raw/scenario31_full`, and
`build_index` recovers the scenario by matching `scenario\d+` against the image
path — so the tree must contain a `scenario31/` component. We symlink rather
than copy: the release is ~9 GB and Kaggle input is read-only but fast.


In [ ]:
DATA = PROJECT / "data" / "raw" / "scenario31_full"
DATA.mkdir(parents=True, exist_ok=True)
link = DATA / "scenario31"
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(RAW)
print("staged", link, "->", RAW)

# convert the per-frame csv into challenge-format windows
out_csv = DATA / "scenario31_challenge_format.csv"
r = subprocess.run([sys.executable, "-u", "preprocessing_amber/adapt_scenario31.py",
                    "--csv", str(RAW / "scenario31_dev.csv"), "--out", str(out_csv)],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, r.stderr[-2000:]


## 4. Preprocess

Radar and LiDAR dominate the runtime. `--source scenario31_full` keeps this off
the existing scenarios, whose tensors we are not rebuilding.


In [ ]:
import time
def run(label, *args):
    t0 = time.time()
    print(f"\n=== {label} ===", flush=True)
    r = subprocess.run([sys.executable, "-u", *args], text=True)
    print(f"--- {label}: {(time.time()-t0)/60:.1f} min, rc={r.returncode}", flush=True)
    return r.returncode

run("radar RA/RV", "preprocessing_amber/radar_ra_rv.py", "--source", "scenario31_full")
run("lidar BEV",   "preprocessing_amber/lidar_bev.py",   "--source", "scenario31_full")
run("image cache", "preprocessing_amber/image_cache.py", "--source", "scenario31_full")


## 5. Index

**The GPS normalisation constants must be the ones the models were trained
under.** There is no train split here to fit on, so refitting — or leaving them
empty, which is what the unpatched code does — would put GPS on a different
scale and silently invalidate every number measured against those weights. We
reuse the originals and report how much of scenario 31 falls outside the
training range, since a geographically distant scenario will clip.


In [ ]:
INDEX_SRC = None
for c in Path("/kaggle/input").rglob("gps_norm.json"):
    INDEX_SRC = c
    break
assert INDEX_SRC, ("no gps_norm.json in the inputs -- attach the preprocessed "
                   "dataset so the training-time GPS constants can be reused")
print("reusing GPS constants from", INDEX_SRC)

run("index", "preprocessing_amber/build_index.py", "--gps-norm-from", str(INDEX_SRC))

## 6. Verify and save

In [ ]:
import pandas as pd, shutil, json
IDX = PROJECT / "data" / "processed_amber" / "index"
s = pd.read_csv(IDX / "samples.csv")
print("rows:", len(s))
print(s.groupby(["split", "scenario"]).size().to_string())
print("\navailability:")
for m in ("image", "lidar", "radar", "gps"):
    c = f"m_{m}"
    if c in s.columns:
        print(f"  {m:<8}{s[c].mean():.3f}")
assert (s["split"] == "scenario31_full").all(), \
    "scenario 31 must be held out whole -- nothing here may be train/val"
assert len(s) > 6000, f"expected ~6804 samples, got {len(s)}"

OUT = Path("/kaggle/working/scenario31_preprocessed")
if OUT.exists():
    shutil.rmtree(OUT)
shutil.copytree(PROJECT / "data" / "processed_amber", OUT,
                symlinks=False, ignore=shutil.ignore_patterns("*.tmp"))
tot = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
print(f"\nsaved {OUT}  ({tot/1e9:.2f} GB)")
print("Release this as a dataset, then attach it to scenario31_evaluate.ipynb")